# mod-BAM Position Mapping Verification

测试HMM pipeline输出的 `p_mod_hmm` 对应的genomic position，是否能通过原始BAM的
`get_aligned_pairs()` 映射到有效的query (read) position。

这是写标准mod-BAM格式（MM/ML tags）的前提条件——MM/ML需要按query position索引的per-base概率。

**检验内容：**
- 从HMM结果里，收集每个read的所有 `(genomic_pos, p_mod_hmm)` 对
- 在原始BAM里找到该read，调用 `get_aligned_pairs()` 获取 `ref_pos → query_pos` 映射
- 检查每个genomic position是否能映射到有效query position（None = deletion位点）
- 统计：映射成功率、失败原因、per-read和per-position分布

## 0. 路径配置

In [ ]:
import logging
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pysam
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

TESTDATA = Path("../testdata").resolve()

NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"

IVT_BAM   = TESTDATA / "control_0.bam"
IVT_FASTQ = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5 = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"

REF_FASTA = TESTDATA / "ref.fa"

for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"  {status}  {label:15s}  {path}")

## 1. 运行pipeline获取HMM结果

`min_depth=1` 让小数据集也能通过depth filter。

In [ ]:
from baleen import run_pipeline_streaming

hmm_results, sites, meta = run_pipeline_streaming(
    native_bam=NATIVE_BAM,
    native_fastq=NATIVE_FASTQ,
    native_blow5=NATIVE_BLOW5,
    ivt_bam=IVT_BAM,
    ivt_fastq=IVT_FASTQ,
    ivt_blow5=IVT_BLOW5,
    ref_fasta=REF_FASTA,
    min_depth=1,
    threads=4,
    run_hmm=True,
)

print(f"处理的contig: {list(hmm_results.keys())}")
for contig, cmr in hmm_results.items():
    n_pos     = len(cmr.position_stats)
    n_native  = len(cmr.native_trajectories)
    n_ivt     = len(cmr.ivt_trajectories)
    print(f"  {contig}: {n_pos} positions, {n_native} native reads, {n_ivt} IVT reads")

## 2. 构建 read → positions 的反向索引

把per-position的HMM结果倒转：对每个read name，收集它出现的所有 `(contig, genomic_pos, p_mod_hmm)`。

In [ ]:
# read_name → list of (contig, genomic_pos, p_mod_hmm, is_native)
read_positions: dict[str, list[tuple]] = defaultdict(list)

for contig, cmr in hmm_results.items():
    for pos, ps in cmr.position_stats.items():
        for i, name in enumerate(ps.native_read_names):
            p = float(ps.p_mod_hmm[i])
            if not np.isnan(p):
                read_positions[name].append((contig, pos, p, True))

        for j, name in enumerate(ps.ivt_read_names):
            p = float(ps.p_mod_hmm[ps.n_native + j])
            if not np.isnan(p):
                read_positions[name].append((contig, pos, p, False))

total_reads   = len(read_positions)
total_entries = sum(len(v) for v in read_positions.values())
print(f"有p_mod_hmm的read数:          {total_reads}")
print(f"(read, position)总条目数:     {total_entries}")
print(f"每个read平均覆盖position数:   {total_entries / total_reads:.1f}")

## 3. 用 `get_aligned_pairs()` 做 genomic pos → query pos 映射

在原始BAM里找到每个read，查它的 `ref_pos → query_pos` 映射表，
看每个HMM position是否能落到一个有效的query position上。

**可能失败的情况：**
- `query_pos is None` → 该位点在CIGAR的D（deletion）操作里，read在这里没有碱基
- position不在aligned_pairs里 → 位点在read的比对范围外（理论上不应出现）

**坐标系注意：** eventalign的position是1-based，pysam的reference_start是0-based，需要 `-1` 转换。

## 3. p_mod_hmm数量 vs. read长度的关系

**核心问题：** 每个read得到的`p_mod_hmm`数量，到底和哪个长度相关？

三个候选：
- `seq_len`：read的总碱基数（含soft clip）
- `n_query_consuming`：CIGAR中M+I+=/X操作消耗的query碱基数（不含soft clip）
- `n_ref_consuming`：CIGAR中M+D+N消耗的reference位置数（即比对跨越的ref范围）

预期：三者都不相等——`n_p_mod`只取决于pipeline里**有足够coverage且通过过滤的position数**，
是这三者的一个稀疏子集。但需要验证具体的比例和分布。

In [ ]:
import re

# CIGAR操作分类
# query-consuming: M(0) I(1) =(7) X(8)
# ref-consuming:   M(0) D(2) N(3) =(7) X(8)
QUERY_CONSUMING = {0, 1, 7, 8}
REF_CONSUMING   = {0, 2, 3, 7, 8}

length_records = []  # per-read长度信息

def scan_bam_lengths(bam_path: Path, is_native: bool):
    with pysam.AlignmentFile(str(bam_path), "rb") as bam:
        for read in bam.fetch():
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue
            name = read.query_name
            if name not in read_positions:
                continue

            n_q = sum(length for op, length in read.cigartuples if op in QUERY_CONSUMING)
            n_r = sum(length for op, length in read.cigartuples if op in REF_CONSUMING)

            # p_mod_hmm条目数（当前read在HMM结果里的position数）
            n_p_mod = sum(
                1 for (_, _, _, pos_is_native) in read_positions[name]
                if pos_is_native == is_native
            )

            length_records.append({
                "read_name":         name,
                "is_native":         is_native,
                "seq_len":           read.query_length,       # 含soft clip
                "n_query_consuming": n_q,                     # M+I+=/X
                "n_ref_consuming":   n_r,                     # M+D+N (ref跨越范围)
                "n_p_mod":           n_p_mod,                 # HMM结果中的position数
            })

scan_bam_lengths(NATIVE_BAM, is_native=True)
scan_bam_lengths(IVT_BAM,    is_native=False)

ldf = pd.DataFrame(length_records)
print(f"统计的read数: {len(ldf)}")
print()
print(ldf[["seq_len", "n_query_consuming", "n_ref_consuming", "n_p_mod"]].describe().round(1))

In [ ]:
# 直接比较：n_p_mod vs 三种长度
print("=== n_p_mod == seq_len 的read数 ===")
eq_seq = (ldf["n_p_mod"] == ldf["seq_len"]).sum()
print(f"  {eq_seq}/{len(ldf)}  ({100*eq_seq/len(ldf):.1f}%)")

print("\n=== n_p_mod == n_query_consuming 的read数 ===")
eq_q = (ldf["n_p_mod"] == ldf["n_query_consuming"]).sum()
print(f"  {eq_q}/{len(ldf)}  ({100*eq_q/len(ldf):.1f}%)")

print("\n=== n_p_mod == n_ref_consuming 的read数 ===")
eq_r = (ldf["n_p_mod"] == ldf["n_ref_consuming"]).sum()
print(f"  {eq_r}/{len(ldf)}  ({100*eq_r/len(ldf):.1f}%)")

print("\n=== n_p_mod vs 各长度的比值（中位数） ===")
print(f"  n_p_mod / seq_len:           {(ldf['n_p_mod'] / ldf['seq_len']).median():.3f}")
print(f"  n_p_mod / n_query_consuming: {(ldf['n_p_mod'] / ldf['n_query_consuming']).median():.3f}")
print(f"  n_p_mod / n_ref_consuming:   {(ldf['n_p_mod'] / ldf['n_ref_consuming']).median():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [
    ("seq_len",           "seq_len\n(含soft clip)"),
    ("n_query_consuming", "n_query_consuming\n(M+I+=/X)"),
    ("n_ref_consuming",   "n_ref_consuming\n(M+D+N, ref跨越范围)"),
]

for ax, (col, label) in zip(axes, pairs):
    ax.scatter(ldf[col], ldf["n_p_mod"], alpha=0.5, s=10)
    # 对角线（相等）
    vmax = max(ldf[col].max(), ldf["n_p_mod"].max())
    ax.plot([0, vmax], [0, vmax], "r--", linewidth=1, label="y=x（相等）")
    ax.set_xlabel(label)
    ax.set_ylabel("n_p_mod")
    ax.set_title(f"n_p_mod  vs  {col}")
    ax.legend(fontsize=8)

plt.suptitle("p_mod_hmm位点数 vs. 各种read长度定义", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
records = []  # 每行对应一个 (read_name, genomic_pos)

def scan_bam(bam_path: Path, is_native: bool):
    with pysam.AlignmentFile(str(bam_path), "rb") as bam:
        for read in bam.fetch():
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue
            name = read.query_name
            if name not in read_positions:
                continue

            # get_aligned_pairs() 返回 (query_pos, ref_pos)，ref_pos是0-based
            # ref=None 的条目是insertion（read有碱基但ref没有），跳过
            ref_to_query: dict[int, int | None] = {
                ref: qry
                for qry, ref in read.get_aligned_pairs()
                if ref is not None
            }

            seq_len = read.query_length

            for contig, genomic_pos, p_mod, pos_is_native in read_positions[name]:
                if pos_is_native != is_native:
                    continue

                # eventalign position是1-based → 转0-based
                ref_pos_0 = genomic_pos - 1

                if ref_pos_0 not in ref_to_query:
                    status = "outside_alignment"
                    query_pos = None
                else:
                    query_pos = ref_to_query[ref_pos_0]
                    status = "deletion" if query_pos is None else "mapped"

                records.append({
                    "read_name":   name,
                    "contig":      contig,
                    "genomic_pos": genomic_pos,
                    "query_pos":   query_pos,
                    "seq_len":     seq_len,
                    "p_mod_hmm":   p_mod,
                    "is_native":   is_native,
                    "status":      status,
                })

scan_bam(NATIVE_BAM, is_native=True)
scan_bam(IVT_BAM,    is_native=False)

df = pd.DataFrame(records)
print(f"扫描的(read, position)总数: {len(df)}")
print()
print("状态分布:")
print(df["status"].value_counts())

## 4. 映射统计

In [ ]:
n_total    = len(df)
n_mapped   = (df["status"] == "mapped").sum()
n_deletion = (df["status"] == "deletion").sum()
n_outside  = (df["status"] == "outside_alignment").sum()
n_missing  = total_entries - n_total  # 在HMM结果里有但BAM里找不到该read

print(f"{'成功映射 (query_pos有效)':<35s} {n_mapped:6d}  ({100*n_mapped/n_total:.2f}%)")
print(f"{'Deletion位点 (query_pos=None)':<35s} {n_deletion:6d}  ({100*n_deletion/n_total:.2f}%)")
print(f"{'比对范围外':<35s} {n_outside:6d}  ({100*n_outside/n_total:.2f}%)")
print(f"{'BAM里找不到该read':<35s} {n_missing:6d}")
print()

# 每个read：是否所有position都映射成功
per_read = df.groupby("read_name")["status"].apply(
    lambda s: (s == "mapped").all()
).rename("all_mapped")
n_reads_full  = per_read.sum()
n_reads_total = len(per_read)
print(f"所有position都映射成功的read: {n_reads_full}/{n_reads_total}  "
      f"({100*n_reads_full/n_reads_total:.1f}%)")
print(f"有至少1个position未映射的read: {n_reads_total - n_reads_full}/{n_reads_total}")

In [ ]:
per_read_counts = df.groupby(["read_name", "status"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：每个read有多少个有效p_mod_hmm的position
n_pos_per_read = df.groupby("read_name").size()
axes[0].hist(n_pos_per_read, bins=30, edgecolor="white")
axes[0].set_xlabel("每个read的分析position数")
axes[0].set_ylabel("reads数")
axes[0].set_title("每个read的p_mod_hmm位点数分布")

# 右图：每个read的映射成功率
frac_mapped = (
    per_read_counts.get("mapped", pd.Series(0, index=per_read_counts.index))
    / per_read_counts.sum(axis=1)
)
axes[1].hist(frac_mapped, bins=20, range=(0, 1), edgecolor="white")
axes[1].set_xlabel("映射成功的position比例")
axes[1].set_ylabel("reads数")
axes[1].set_title("per-read映射成功率分布")

plt.tight_layout()
plt.show()

## 5. 单read抽查

挑一个position最多的native read，打印完整的映射表，验证query_pos的范围。

In [ ]:
native_df    = df[df["is_native"]]
example_read = native_df.groupby("read_name").size().idxmax()
sub          = native_df[native_df["read_name"] == example_read].sort_values("genomic_pos")

print(f"Read:    {example_read}")
print(f"seq_len: {sub['seq_len'].iloc[0]}")
print(f"分析的position数: {len(sub)}")
print()
print(sub[["contig", "genomic_pos", "query_pos", "p_mod_hmm", "status"]].to_string(index=False))

In [ ]:
mapped_sub = sub[sub["status"] == "mapped"]
seq_len    = int(sub["seq_len"].iloc[0])

oob = mapped_sub[mapped_sub["query_pos"] >= seq_len]
print(f"query_pos越界 (>= seq_len={seq_len}): {len(oob)}")

if len(oob) == 0:
    print("✅  该read所有query_pos都是合法索引")
else:
    print("❌  存在越界的query_pos！")
    print(oob)

## 6. 模拟MM/ML tag构造

对上面那个example read，构造 `MM:Z:` 和 `ML:B:C` tag的内容。

使用 `N+?`（未知修饰类型，所有碱基都作为候选位点）。

**MM的delta编码规则：**
- 把所有有修饰的query position排序
- 每个位置记录跳过的碱基数（从上一个修饰位置到当前位置之间的碱基数，不含两端）
- 例：query positions [5, 10, 15] → skip counts [5, 4, 4] → `N+?,5,4,4;`

In [ ]:
def build_mm_ml_tags(
    query_positions: list[int],
    probabilities: list[float],
) -> tuple[str, list[int]]:
    """为N+?（未知修饰）构造MM字符串和ML uint8数组。

    Parameters
    ----------
    query_positions : sorted list of 0-based query positions
    probabilities   : 对应的p_mod值，范围[0, 1]

    Returns
    -------
    mm_str  : MM:Z tag的值
    ml_vals : ML:B:C tag的uint8数组
    """
    assert len(query_positions) == len(probabilities)
    assert query_positions == sorted(query_positions), "query_positions必须已排序"

    deltas = []
    prev = -1
    for qp in query_positions:
        deltas.append(qp - prev - 1)
        prev = qp

    mm_str  = "N+?," + ",".join(str(d) for d in deltas) + ";"
    ml_vals = [min(255, round(p * 255)) for p in probabilities]
    return mm_str, ml_vals


mapped_sorted = mapped_sub.sort_values("query_pos")
qpos  = mapped_sorted["query_pos"].tolist()
probs = mapped_sorted["p_mod_hmm"].tolist()

mm_str, ml_vals = build_mm_ml_tags(qpos, probs)

print(f"tagged positions数: {len(qpos)}")
print(f"MM:Z  {mm_str}")
print(f"ML:B:C  {ml_vals[:20]}{'...' if len(ml_vals) > 20 else ''}")

## 7. 全局合法性检验

In [ ]:
mapped_df = df[df["status"] == "mapped"].copy()

oob   = mapped_df[mapped_df["query_pos"] >= mapped_df["seq_len"]]
neg   = mapped_df[mapped_df["query_pos"] < 0]
bad_p = mapped_df[(mapped_df["p_mod_hmm"] < 0) | (mapped_df["p_mod_hmm"] > 1)]

print(f"query_pos越界 (>= seq_len):   {len(oob)}")
print(f"query_pos为负数:              {len(neg)}")
print(f"p_mod_hmm超出[0,1]范围:       {len(bad_p)}")

if len(oob) == 0 and len(neg) == 0 and len(bad_p) == 0:
    print()
    print("✅  所有映射成功的position都可以安全写入MM/ML tag")
    print(f"   （{n_mapped}个position可以写入；"
          f"{n_deletion + n_outside}个deletion/范围外的position跳过）")
else:
    print("\n❌  发现问题，见上方详情")